# SVD

# DEFINING MODEL

In [11]:
import pandas as pd
import numpy as np
import re
from sklearn.decomposition import TruncatedSVD
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity

In [12]:
# Only adjust the CLUSTER, WEIGHT_SVD, WEIGHT_DNN, default_weights, DATA_PATH, PLAYLIST_FILE, TRACKS_FILE

# ========== GLOBAL CONFIGURATION ==========
CLUSTER = None

DATA_PATH = "/Users/xavierhua/Documents/GitHub/spotifynd/phase5_model_development"
PLAYLIST_FILE = f"{DATA_PATH}/playlist_final_final.csv"
TRACKS_FILE = f"{DATA_PATH}/tracks_new.csv"
NUM_PLAYLISTS = 1000
K_EVAL = 50
LATENT_DIM = 100

In [13]:
###################################
# 1. DATA LOADING & PREPROCESSING (SVD)
###################################
def load_data():
    playlists = pd.read_csv(PLAYLIST_FILE, engine='python', on_bad_lines='skip')
    tracks = pd.read_csv(TRACKS_FILE)
    if CLUSTER:
        playlists = playlists[playlists['cluster'] == CLUSTER].reset_index(drop=True)[:NUM_PLAYLISTS]
    else:
        playlists = playlists[:NUM_PLAYLISTS]
    return playlists, tracks

def clean_centroid_string(s):
    """Converts a centroid string into a list of floats."""
    if isinstance(s, str):
        s = re.sub(r'[\[\]]', '', s).strip()
        return [float(x) for x in s.split()]
    return s

def parse_track_list(s):
    """Extracts integers from a string and returns them as a list."""
    return [int(x) for x in re.findall(r'\d+', s)]

def preprocess_playlists(df):
    """Preprocesses the playlist DataFrame by cleaning centroids and parsing track lists."""
    df = df.copy()
    for col in ['sentiment_centroid', 'genre_centroid']:
        if col in df.columns:
            df[col] = df[col].apply(clean_centroid_string)
    for col in ['track_idx_list', 'tracks_to_predict']:
        if col in df.columns:
            df[col] = df[col].apply(parse_track_list)
    return df

In [14]:
###################################
# 2. DATA SPLITTING & INTERACTION MATRIX (SVD)
###################################
def split_data(playlists):
    """
    Splits playlists into train, validation, and test sets
    based on the 'dataset_type' column.
    """
    train = playlists[playlists['dataset_type'] == 'train'].reset_index(drop=True)
    val = playlists[playlists['dataset_type'] == 'val'].reset_index(drop=True)
    test = playlists[playlists['dataset_type'] == 'test'].reset_index(drop=True)
    final = playlists[playlists['dataset_type'] == 'final'].reset_index(drop=True)
    return train, val, test, final

def build_track_mapping(train_playlists, tracks_df):
    """Creates a sorted list of all available tracks and a mapping to column indices."""
    unique_tracks = sorted(tracks_df['track_idx'].unique().tolist())
    track_to_col = {track: idx for idx, track in enumerate(unique_tracks)}
    return unique_tracks, track_to_col

def build_interaction_matrix(playlists_subset, track_to_col, track_col='track_idx_list'):
    """Builds a binary interaction matrix for the given subset of playlists."""
    n = len(playlists_subset)
    m = len(track_to_col)
    matrix = np.zeros((n, m), dtype=np.int32)
    for i, row in playlists_subset.iterrows():
        for track in row[track_col]:
            if track in track_to_col:
                matrix[i, track_to_col[track]] = 1
    return matrix

def create_track_feat_map_svd(tracks_df):
    """
    Creates a feature mapping for tracks using the full tracks DataFrame.
    """
    feat_cols = ['track_popularity',
                 'Early Years', 'Classic Era', 'Golden Era', '2000s', 'Modern Era',
                 'Short', 'Medium', 'Long',
                 'joy', 'calm', 'sadness', 'fear', 'energizing', 'dreamy',
                 'Instrumental / Ambient Sounds', 'Soft Acoustic / Classical', 'Orchestral / Soundtrack',
                 'Mid-tempo Pop / Indie', 'Upbeat Electronic / Dance', 'Slow & Melancholic (Sad Songs)',
                 'Experimental / Jazz Fusion', 'Lo-Fi / Chill Vibes']
    feat_map = {row['track_idx']: row[feat_cols].values for _, row in tracks_df.iterrows()}
    return feat_map

In [15]:
###################################
# 3. SVD MODEL TRAINING, PREDICTION & EVALUATION (SVD)
###################################
def train_svd_model(interaction_matrix_train):
    """Trains an SVD model on the training interaction matrix."""
    svd = TruncatedSVD(n_components=LATENT_DIM, random_state=42)
    U_train = svd.fit_transform(interaction_matrix_train)
    Sigma = svd.singular_values_
    VT = svd.components_
    sqrt_sigma = np.sqrt(Sigma)
    P_train = U_train * sqrt_sigma  # Playlist latent factors
    Q = (VT.T * sqrt_sigma)         # Track latent factors
    return svd, P_train, Q, sqrt_sigma

def fold_in_playlists(svd, interaction_matrix, sqrt_sigma):
    """Folds in new playlists into the SVD latent space."""
    return svd.transform(interaction_matrix) * sqrt_sigma

def predict_mf(playlist_latent, interaction_row, Q):
    """
    Predicts scores for all tracks for a given playlist using its latent factors.
    Existing tracks are excluded by setting their scores to -∞.
    """
    scores = playlist_latent.dot(Q.T)
    existing = np.where(interaction_row > 0)[0]
    scores[existing] = -np.inf
    return scores

def predict_mf_wrapper(playlist_idx, interaction_matrix, P_matrix, Q):
    """Wrapper to predict scores for a playlist by its index."""
    return predict_mf(P_matrix[playlist_idx], interaction_matrix[playlist_idx], Q)

def get_all_svd_predictions(interaction_matrix, P_matrix, predict_func, Q, test_playlists):
    """
    Returns a dictionary mapping the original playlist IDs (from test_playlists) 
    to the predicted score vector for each playlist.
    """
    predictions = {}
    n = interaction_matrix.shape[0]
    for i in range(n):
        # Get the original playlist ID
        playlist_id = test_playlists.iloc[i]['playlist_idx']
        predictions[playlist_id] = predict_func(i, interaction_matrix, P_matrix, Q)
    return predictions

def compute_metrics_for_playlist(predicted_scores, test_indices, k=10):
    """
    Computes Hit@k, MRR, and MAP for a given playlist.
    """
    ranked_indices = np.argsort(-predicted_scores)
    top_k = ranked_indices[:k]
    hit = 1 if any(t in top_k for t in test_indices) else 0
    precisions = []
    num_hits = 0
    mrr = 0.0
    for rank_idx, track_idx in enumerate(ranked_indices[:k]):
        if track_idx in test_indices:
            num_hits += 1
            precisions.append(num_hits / (rank_idx + 1))
            if mrr == 0.0:
                mrr = 1.0 / (rank_idx + 1)
    ap = np.mean(precisions) if precisions else 0.0
    return hit, mrr, ap

def build_ground_truth(playlists_subset, track_to_col, track_col='tracks_to_predict'):
    """
    Builds a dictionary mapping each playlist (using its original 'playlist_idx')
    to a list of ground truth track indices.
    """
    test_items = {}
    for _, row in playlists_subset.iterrows():
        pid = row['playlist_idx']
        test_items[pid] = [track_to_col[t] for t in row[track_col] if t in track_to_col]
    return test_items

def evaluate_model_svd(interaction_matrix, ground_truth, P_matrix, predict_func, Q, playlistid_to_index, k=50):
    """
    Evaluates the SVD model using ground truth keyed by original playlist IDs.
    It returns only Hit@K, MRR, and MAP@K.
    """
    hit_total, mrr_total, ap_total = 0, 0, 0
    n = len(ground_truth)
    for pid, true_indices in ground_truth.items():
        if pid not in playlistid_to_index:
            continue  # skip if the playlist ID is not found in the test data
        idx = playlistid_to_index[pid]
        predicted_scores = predict_func(idx, interaction_matrix, P_matrix, Q)
        hit, mrr, ap = compute_metrics_for_playlist(predicted_scores, true_indices, k)
        hit_total += hit
        mrr_total += mrr
        ap_total += ap
    return {
        'Hit@K': hit_total / n,
        'MRR': mrr_total / n,
        'MAP@K': ap_total / n
    }

# TRAIN

In [ ]:
# Load and preprocess data
playlist_raw, tracks = load_data()
playlists = preprocess_playlists(playlist_raw)

# Split data into train, validation, test, and final sets
train_playlists, val_playlists, test_playlists, final_playlists = split_data(playlists)

# Build track mapping (based on training data) and interaction matrices
unique_tracks, track_to_col = build_track_mapping(train_playlists, tracks)
print("Training playlists:", len(train_playlists))
print("Unique tracks (from train):", len(unique_tracks))

interaction_matrix_train = build_interaction_matrix(train_playlists, track_to_col)
interaction_matrix_val = build_interaction_matrix(val_playlists, track_to_col)
interaction_matrix_test = build_interaction_matrix(test_playlists, track_to_col)
interaction_matrix_final = build_interaction_matrix(final_playlists, track_to_col)
print("Training interaction matrix shape:", interaction_matrix_train.shape)
print("Validation interaction matrix shape:", interaction_matrix_val.shape)
print("Test interaction matrix shape:", interaction_matrix_test.shape)
print("Final interaction matrix shape:", interaction_matrix_final.shape)

track_feat_map = create_track_feat_map_svd(tracks)

# Train SVD model on training set
svd, P_train, Q, sqrt_sigma = train_svd_model(interaction_matrix_train)

Training playlists: 665
Unique tracks (from train): 250426
Training interaction matrix shape: (665, 250426)
Validation interaction matrix shape: (98, 250426)
Test interaction matrix shape: (104, 250426)
Final interaction matrix shape: (133, 250426)


# VALIDATE

In [17]:
# Fold in validation playlists
P_val = fold_in_playlists(svd, interaction_matrix_val, sqrt_sigma)
# Get SVD predictions on validation using original playlist IDs as keys
svd_predictions_val = get_all_svd_predictions(interaction_matrix_val, P_val, predict_mf_wrapper, Q, val_playlists)
# Build ground truth for validation (keys are original playlist IDs)
ground_truth_val = build_ground_truth(val_playlists, track_to_col)
# Build mapping from original playlist IDs to row indices for the validation set
playlistid_to_index_val = {row['playlist_idx']: i for i, row in val_playlists.iterrows()}
# Evaluate on validation set
svd_metrics_val = evaluate_model_svd(interaction_matrix_val, ground_truth_val, P_val, predict_mf_wrapper, Q, playlistid_to_index_val, k=K_EVAL)
print("SVD Evaluation on Validation:")
print(f"Hit@{K_EVAL}: {svd_metrics_val['Hit@K']:.4f}")
print(f"MRR:         {svd_metrics_val['MRR']:.4f}")
print(f"MAP@{K_EVAL}: {svd_metrics_val['MAP@K']:.4f}")

# Convert validation recommendations to DataFrame
index_to_track = {v: k for k, v in track_to_col.items()}
val_recommendations = []
n_val = interaction_matrix_val.shape[0]
for i in range(n_val):
    scores = predict_mf_wrapper(i, interaction_matrix_val, P_val, Q)
    ranked_indices = np.argsort(-scores)
    top_indices = ranked_indices[:K_EVAL]
    recommended_tracks = [index_to_track[idx] for idx in top_indices if idx in index_to_track]
    playlist_id = val_playlists.iloc[i]['playlist_idx']
    val_recommendations.append({'playlist_idx': playlist_id, 'recommended_tracks': recommended_tracks})
val_rec_df = pd.DataFrame(val_recommendations)
print(val_rec_df.head())


SVD Evaluation on Validation:
Hit@50: 0.5612
MRR:         0.1279
MAP@50: 0.0937
   playlist_idx                                 recommended_tracks
0             4  [96437, 178396, 211269, 75113, 81826, 159442, ...
1             6  [86790, 250306, 235367, 237495, 129996, 64550,...
2            12  [237495, 14975, 203557, 16769, 216374, 86790, ...
3            49  [59031, 249785, 246233, 189533, 192501, 143682...
4            66  [250306, 64550, 50047, 221079, 235367, 216374,...


# TEST

In [18]:
# Fold in test playlists
P_test = fold_in_playlists(svd, interaction_matrix_test, sqrt_sigma)
# Get SVD predictions on test using original playlist IDs as keys
svd_predictions_test = get_all_svd_predictions(interaction_matrix_test, P_test, predict_mf_wrapper, Q, test_playlists)
# Build ground truth for test
ground_truth_test = build_ground_truth(test_playlists, track_to_col)
# Build mapping from original playlist IDs to row indices for the test set
playlistid_to_index_test = {row['playlist_idx']: i for i, row in test_playlists.iterrows()}
# Evaluate on test set
svd_metrics_test = evaluate_model_svd(interaction_matrix_test, ground_truth_test, P_test, predict_mf_wrapper, Q, playlistid_to_index_test, k=K_EVAL)
print("SVD Evaluation on Test:")
print(f"Hit@{K_EVAL}: {svd_metrics_test['Hit@K']:.4f}")
print(f"MRR:         {svd_metrics_test['MRR']:.4f}")
print(f"MAP@{K_EVAL}: {svd_metrics_test['MAP@K']:.4f}")

# Convert test recommendations to DataFrame
test_recommendations = []
n_test = interaction_matrix_test.shape[0]
for i in range(n_test):
    scores = predict_mf_wrapper(i, interaction_matrix_test, P_test, Q)
    ranked_indices = np.argsort(-scores)
    top_indices = ranked_indices[:K_EVAL]
    recommended_tracks = [index_to_track[idx] for idx in top_indices if idx in index_to_track]
    playlist_id = test_playlists.iloc[i]['playlist_idx']
    test_recommendations.append({'playlist_idx': playlist_id, 'recommended_tracks': recommended_tracks})
test_rec_df = pd.DataFrame(test_recommendations)
print(test_rec_df.head())

SVD Evaluation on Test:
Hit@50: 0.5288
MRR:         0.1156
MAP@50: 0.0801
   playlist_idx                                 recommended_tracks
0             2  [73995, 66741, 158749, 58141, 231254, 102468, ...
1             7  [191313, 168972, 44749, 216260, 79687, 163566,...
2            10  [208248, 53989, 30287, 218307, 221104, 148651,...
3            17  [235367, 216374, 64550, 14975, 54047, 237415, ...
4            28  [250306, 86790, 64550, 180201, 191177, 26659, ...
